In [2]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database connection
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create tables
cursor.execute('''
CREATE TABLE Customers (
CustomerID     INT,
CustomerName   VARCHAR(50),
City           VARCHAR(30),
JoinDate       DATE
);
''')

cursor.execute('''
CREATE TABLE Orders (
OrderID      INT,
CustomerID   INT,
OrderDate    DATE,
OrderAmount  DECIMAL(10,2),
Status VARCHAR(20)
);
''')

cursor.execute('''
CREATE TABLE Employees (
EmployeeID    INT,
EmployeeName  VARCHAR(50),
ManagerID     INT,
Department    VARCHAR(30),
JoiningDate   DATE,
Salary        DECIMAL(10,2)
);
''')

# Insert data
customers_data = [
    (101,'Rahul Sharma','Bangalore','2023-01-15'),
    (102,'Priya Mehta','Mumbai','2024-03-20'),
    (103,'Arjun Nair','Chennai','2025-01-12'),
    (104,'Sneha Gupta','Delhi','2022-07-10'),
    (105,'Kavya Reddy','Hyderabad','2024-08-15'),
    (106,'Amit Verma','Pune','2023-11-05'),
    (107,'Rohan Singh','Kolkata','2024-05-18'),
    (108,'Simran Kaur','Delhi','2025-02-01'),
    (109,'Anjali Jain','Mumbai','2023-09-22'),
    (110,'Vikram Patel','Ahmedabad','2022-12-30')
]
cursor.executemany("INSERT INTO Customers VALUES (?,?,?,?)", customers_data)

orders_data = [
    (5001,101,'2025-01-05',12000,'Delivered'),
    (5002,102,'2025-01-15',8500,'Pending'),
    (5003,101,'2025-02-10',22000,'Delivered'),
    (5004,104,'2025-02-15',4500,'Cancelled'),
    (5005,103,'2025-03-01',17500,'Delivered'),
    (5006,105,'2025-03-10',9800,'Pending'),
    (5007,106,'2025-03-18',25000,'Delivered'),
    (5008,107,'2025-04-02',14500,'Delivered'),
    (5009,101,'2025-04-10',30000,'Delivered'),
    (5010,109,'2025-04-15',6500,'Pending'),
    (5011,110,'2025-05-01',42000,'Delivered'),
    (5012,104,'2025-05-08',18000,'Delivered'),
    (5013,106,'2025-05-15',7000,'Cancelled'),
    (5014,102,'2025-05-20',27000,'Delivered'),
    (5015,107,'2025-06-01',15500,'Pending')
]
cursor.executemany("INSERT INTO Orders VALUES (?,?,?,?,?)", orders_data)

employees_data = [
    (1,'Rajesh Kumar',None,'Management','2018-01-10',150000),
    (2,'Neha Sharma',1,'Sales','2020-03-15',90000),
    (3,'Amit Gupta',1,'IT','2019-06-20',110000),
    (4,'Priyanka Singh',2,'Sales','2022-01-12',65000),
    (5,'Vikas Patel',2,'Sales','2021-09-18',70000),
    (6,'Rohit Jain',3,'IT','2023-02-05',60000),
    (7,'Anjali Verma',3,'IT','2022-11-10',62000),
    (8,'Karan Mehta',1,'HR','2021-05-25',80000),
    (9,'Sneha Kapoor',8,'HR','2024-01-05',50000),
    (10,'Arjun Malhotra',3,'IT','2024-04-15',55000)
]
cursor.executemany("INSERT INTO Employees VALUES (?,?,?,?,?,?)", employees_data)
conn.commit()

# Helper function to run and display queries as Pandas DataFrames
def run_query(title, query):
    print(f"=== {title} ===")
    try:
        df = pd.read_sql_query(query, conn)
        print(df.to_string(index=False))
    except Exception as e:
        print("Error:", e)
    print("\n")

# Query 1: Top 5 orders by order amount with customer names
run_query("Top 5 Orders by Amount", """
SELECT
    o.OrderID,
    c.CustomerName,
    o.OrderAmount,
    o.OrderDate,
    o.Status
FROM Orders o
JOIN Customers c ON o.CustomerID = c.CustomerID
ORDER BY o.OrderAmount DESC
LIMIT 5;
""")

# Query 2: Customers whose name starts with A, R, or S
run_query("Customers starting with A, R, S", """
SELECT CustomerName
FROM Customers
WHERE SUBSTR(CustomerName, 1, 1) IN ('A','R','S');
""")

# Query 3: Pending orders above 10,000
run_query("Pending Orders > 10000", """
SELECT c.CustomerName,
       o.OrderID,
       o.OrderAmount,
       o.Status
FROM Orders o
JOIN Customers c ON o.CustomerID = c.CustomerID
WHERE o.OrderAmount > 10000
  AND o.Status = 'Pending';
""")

# Query 4: Customers associated for more than 365 days
run_query("Customers associated > 365 days", """
SELECT CustomerID,
       CustomerName,
       JoinDate,
       CAST(julianday('now') - julianday(JoinDate) AS INT) AS DayAssociated
FROM Customers
WHERE (julianday('now') - julianday(JoinDate)) > 365;
""")

# Query 5: Tax computation and 15 days addition
run_query("Order values with 18% Tax and 15-day Shipping Estimate", """
SELECT
    OrderID,
    OrderAmount,
    ROUND(OrderAmount*0.18, 2) AS Tax,
    ROUND(OrderAmount*1.18, 2) AS TotalAmount,
    date(OrderDate, '+15 days') AS ProjectedDeliveryDate
FROM Orders;
""")

# Query 6: Offset-based top orders (Rank 11-20)
run_query("Orders 11 to 20 by Amount", """
SELECT c.CustomerID,
       c.CustomerName,
       o.OrderID,
       o.OrderAmount
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
ORDER BY o.OrderAmount DESC
LIMIT 10 OFFSET 10;
""")

# Query 7: Inactive Customers (No Orders placed)
run_query("Customers with no Orders", """
SELECT c.CustomerID,
       c.CustomerName,
       c.JoinDate,
       CAST(julianday('now') - julianday(c.JoinDate) AS INT) AS DaysSinceJoin
FROM Customers c
LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
WHERE o.OrderID IS NULL;
""")

# Query 8: Self-join to display employee manager hierarchy
run_query("Employee Manager Hierarchy", """
SELECT
    e.EmployeeName,
    m.EmployeeName AS ManagerName,
    e.Department
FROM Employees e
LEFT JOIN Employees m ON e.ManagerID = m.EmployeeID;
""")

# Query 9: Customers who ordered within 30 days of joining
run_query("Ordered within 30 Days of Joining", """
SELECT
    c.CustomerName,
    c.JoinDate,
    MIN(o.OrderDate) AS FirstOrderDate,
    CAST(julianday(MIN(o.OrderDate)) - julianday(c.JoinDate) AS INT) AS DaysToFirstOrder
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CustomerName, c.JoinDate
HAVING (julianday(MIN(o.OrderDate)) - julianday(c.JoinDate)) BETWEEN 0 AND 30;
""")

# Query 10: Customer Segments and Loyalty Status
run_query("Customer Spending and Loyalty Segmentation", """
SELECT
    c.CustomerName,
    COALESCE(SUM(o.OrderAmount), 0) AS TotalPurchaseAmount,
    CASE
        WHEN COALESCE(SUM(o.OrderAmount), 0) > 50000 THEN 'Premium'
        WHEN COALESCE(SUM(o.OrderAmount), 0) BETWEEN 20000 AND 50000 THEN 'Gold'
        ELSE 'Silver'
    END AS SpendingCategory,
    CASE
        WHEN c.JoinDate < '2024-01-01' THEN 'Loyal Customer'
        ELSE 'New Customer'
    END AS LoyaltyStatus
FROM Customers c
LEFT JOIN Orders o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CustomerName, c.JoinDate;
""")

# Query 11: Orders greater than the overall average order amount
run_query("Orders Above Average Order Value", """
SELECT c.CustomerName AS 'Customer Name',
       o.OrderID AS 'Order ID',
       o.OrderAmount AS 'Order Amount'
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
WHERE o.OrderAmount > (SELECT AVG(OrderAmount) FROM Orders);
""")

# Query 12: 2nd to 6th highest orders (limit 5 offset 1)
run_query("2nd to 6th Highest Orders", """
SELECT c.CustomerName AS 'Customer Name',
       o.OrderID AS 'Order ID',
       o.OrderAmount AS 'Order Amount'
FROM Customers c
JOIN Orders o ON c.CustomerID = o.CustomerID
ORDER BY o.OrderAmount DESC
LIMIT 5 OFFSET 1;
""")

=== Top 5 Orders by Amount ===
 OrderID CustomerName  OrderAmount  OrderDate    Status
    5011 Vikram Patel        42000 2025-05-01 Delivered
    5009 Rahul Sharma        30000 2025-04-10 Delivered
    5014  Priya Mehta        27000 2025-05-20 Delivered
    5007   Amit Verma        25000 2025-03-18 Delivered
    5003 Rahul Sharma        22000 2025-02-10 Delivered


=== Customers starting with A, R, S ===
CustomerName
Rahul Sharma
  Arjun Nair
 Sneha Gupta
  Amit Verma
 Rohan Singh
 Simran Kaur
 Anjali Jain


=== Pending Orders > 10000 ===
CustomerName  OrderID  OrderAmount  Status
 Rohan Singh     5015        15500 Pending


=== Customers associated > 365 days ===
 CustomerID CustomerName   JoinDate  DayAssociated
        101 Rahul Sharma 2023-01-15           1410
        102  Priya Mehta 2024-03-20            980
        103   Arjun Nair 2025-01-12            682
        104  Sneha Gupta 2022-07-10           1599
        105  Kavya Reddy 2024-08-15            832
        106   Amit V